# 99 - Robustness Checks

Test sensitivity of the main NYC TWFE estimate to: control group composition, outcome specification, anticipation, and alternative event windows.

In [ ]:
# Data setup
# Set DATA_FILE to 'city_month_panel_real.parquet' after running build_real_panel.py
DATA_FILE = "city_month_panel_real.parquet"   # real Inside Airbnb data
# DATA_FILE = "city_month_panel.parquet"       # synthetic fallback (not committed)

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

DATA_DIR = Path("../data/processed")
OUT_DIR  = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

panel = pd.read_parquet(DATA_DIR / DATA_FILE)
panel["month"] = pd.to_datetime(panel["month"])

regs = pd.read_csv("../data/regulations.csv", parse_dates=["enforcement_date"])

print(f"Panel: {panel.shape}  |  Cities: {sorted(panel['city'].unique())}")
print(f"Date range: {panel['month'].min().date()} to {panel['month'].max().date()}")

In [ ]:
from linearmodels.panel import PanelOLS

NYC_TREAT = pd.Timestamp("2023-09-01")
BASE_DATA = panel[panel["city"] != "Florence"].copy()

def twfe(data, outcome="log_listings"):
    d = data.copy()
    d["post"]    = (d["month"] >= NYC_TREAT).astype(float)
    d["treated"] = (d["city"] == "New York City").astype(float)
    d["did"]     = d["treated"] * d["post"]
    idx = d.set_index(["city","month"])
    fe  = PanelOLS(idx[outcome], idx[["did"]],
                   entity_effects=True, time_effects=True).fit(
                   cov_type="clustered", cluster_entity=True)
    b, ci = fe.params["did"], fe.conf_int().loc["did"]
    return b, ci["lower"], ci["upper"]

b_base, lo, hi = twfe(BASE_DATA)
print(f"Baseline ATT (log_listings): {b_base:.4f}  [{lo:.4f}, {hi:.4f}]")

## Check 1 - Leave-one-control-out

In [ ]:
controls = ["Amsterdam","Lisbon","Vienna","Barcelona"]
print(f"{'Spec':<30}  {'ATT':>8}  CI")
print("-" * 55)
print(f"{'All controls':<30}  {b_base:>8.4f}  [{lo:.4f}, {hi:.4f}]")
for drop in controls:
    sub = BASE_DATA[BASE_DATA["city"] != drop]
    b, lo_, hi_ = twfe(sub)
    print(f"{'Drop '+drop:<30}  {b:>8.4f}  [{lo_:.4f}, {hi_:.4f}]")

## Check 2 - Alternative outcomes

In [ ]:
for outcome in ["log_listings","log_price","availability_rate","entire_home_share"]:
    b, lo_, hi_ = twfe(BASE_DATA, outcome)
    print(f"{outcome:<25}  ATT = {b:+.4f}  [{lo_:+.4f}, {hi_:+.4f}]")

## Check 3 - Anticipation: shift treatment date earlier

In [ ]:
for shift_months in [0, -3, -6]:
    shifted_date = NYC_TREAT + pd.DateOffset(months=shift_months)
    sub = BASE_DATA.copy()
    sub["post"]    = (sub["month"] >= shifted_date).astype(float)
    sub["treated"] = (sub["city"] == "New York City").astype(float)
    sub["did"]     = sub["treated"] * sub["post"]
    idx = sub.set_index(["city","month"])
    fe  = PanelOLS(idx["log_listings"], idx[["did"]],
                   entity_effects=True, time_effects=True).fit(
                   cov_type="clustered", cluster_entity=True)
    b  = fe.params["did"]
    ci = fe.conf_int().loc["did"]
    label = f"Treat date = {shifted_date.date()} (shift {shift_months}m)"
    print(f"{label:<40}  ATT = {b:+.4f}  [{ci['lower']:+.4f}, {ci['upper']:+.4f}]")

## Summary

A robust design shows:
- **Stable ATT** across control group compositions
- **Price and availability effects** consistent with a supply shock story
- **Anticipation check**: if shifting date earlier produces similar ATT, some pre-enforcement de-listing occurred
- **Leave-one-out**: no single control city drives the result